In [1]:
import pandas as pd
import time
import torch
import torch.nn.functional as F

In [2]:
df = pd.read_csv(
    'simpsons_script_lines.csv',
    usecols=['normalized_text'],
    low_memory=False
)

phrases = df['normalized_text'].tolist()

text = [[c for c in ph] for ph in phrases if type(ph) is str]

In [3]:
CHARS = set('abcdefghijklmnopqrstuvwxyz ')
INDEX_TO_CHAR = ['none'] + list(CHARS)
CHAR_TO_INDEX = {c: i for i, c in enumerate(INDEX_TO_CHAR)}

VOCAB_SIZE = len(INDEX_TO_CHAR)

In [4]:
MAX_LEN = 30

X = torch.zeros((len(text), MAX_LEN), dtype=torch.long)

for i in range(len(text)):
    for j, c in enumerate(text[i]):
        if j >= MAX_LEN:
            break
        X[i, j] = CHAR_TO_INDEX.get(c, 0)

In [5]:
class MyRNNCell(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.Wx = torch.nn.Linear(input_size, hidden_size)
        self.Wh = torch.nn.Linear(hidden_size, hidden_size)

    def forward(self, x, h):
        return torch.tanh(self.Wx(x) + self.Wh(h))

In [6]:
class MyRNN(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.cell = MyRNNCell(input_size, hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        h = torch.zeros(batch_size, self.hidden_size)

        outputs = []

        for t in range(seq_len):
            h = self.cell(x[:, t, :], h)
            outputs.append(h.unsqueeze(1))

        return torch.cat(outputs, dim=1)

In [7]:
class Network(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(VOCAB_SIZE, 30)
        self.rnn = MyRNN(30, 256)
        self.fc = torch.nn.Linear(256, VOCAB_SIZE)

    def forward(self, x):
        x = self.embedding(x)
        x = self.rnn(x)
        return self.fc(x)

model = Network()

In [8]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in range(30):
    start = time.time()
    total_loss = 0
    steps = 0

    for i in range(0, len(X), 100):
        batch = X[i:i+100]

        X_batch = batch[:, :-1]
        Y_batch = batch[:, 1:].reshape(-1)

        pred = model(X_batch)
        pred = pred.reshape(-1, VOCAB_SIZE)

        loss = criterion(pred, Y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    print(f'epoch {ep}, time {time.time()-start:.2f}, loss {total_loss/steps:.4f}')

epoch 0, time 12.40, loss 1.7113
epoch 1, time 12.22, loss 1.4621
epoch 2, time 13.14, loss 1.4054
epoch 3, time 13.79, loss 1.3750
epoch 4, time 13.71, loss 1.3552
epoch 5, time 12.72, loss 1.3412
epoch 6, time 12.64, loss 1.3306
epoch 7, time 12.76, loss 1.3223
epoch 8, time 12.74, loss 1.3155
epoch 9, time 12.52, loss 1.3098
epoch 10, time 12.59, loss 1.3050
epoch 11, time 12.70, loss 1.3009
epoch 12, time 12.72, loss 1.2973
epoch 13, time 12.58, loss 1.2942
epoch 14, time 12.68, loss 1.2914
epoch 15, time 12.70, loss 1.2890
epoch 16, time 12.68, loss 1.2869
epoch 17, time 13.24, loss 1.2850
epoch 18, time 13.01, loss 1.2833
epoch 19, time 12.68, loss 1.2818
epoch 20, time 12.71, loss 1.2804
epoch 21, time 12.65, loss 1.2791
epoch 22, time 12.66, loss 1.2780
epoch 23, time 12.54, loss 1.2770
epoch 24, time 12.64, loss 1.2761
epoch 25, time 12.65, loss 1.2752
epoch 26, time 12.51, loss 1.2745
epoch 27, time 12.72, loss 1.2738
epoch 28, time 12.57, loss 1.2732
epoch 29, time 12.54, lo

In [9]:
def generate(start_text, max_len=100, temperature=1.0):
    model.eval()

    chars = list(start_text)
    x = torch.tensor([[CHAR_TO_INDEX.get(c, 0) for c in chars]])

    for _ in range(max_len):
        out = model(x)
        logits = out[0, -1] / temperature

        probs = F.softmax(logits, dim=0)
        next_char = torch.multinomial(probs, 1).item()

        chars.append(INDEX_TO_CHAR[next_char])
        x = torch.tensor([[CHAR_TO_INDEX.get(c, 0) for c in chars]])

    return ''.join(chars)

In [10]:
print(generate('homer '))
print(generate('bart '))
print(generate('lisa '))

homer i made the fifty order this is reverach hes buying stop homer you know i told you lisa simpson right
bart hey you bear this delimission how mantures gonna stay of you any denisuive indow boy untairrgical ni
lisa i chall your son it award you begins keen handre is not mastery problemm room inaliration marge sorr
